# JSON 数据的语义检索（JSON RAG）

目标：把 JSON 这种结构化数据转换为“更适合向量化检索的文本表示”，再用 embedding + 向量索引完成**按语义**而不是按关键词的搜索。


## 为什么需要这一步？

大多数语义检索模型擅长处理“文本”，但 JSON 通常包含大量结构字段。直接把整个 JSON 扁平化为大字符串，会引入噪声，影响 embedding 质量。

本节使用 `jrag`（基于 `jsonpath-ng`）来**精确挑选关键字段**，把 JSON 记录转换为一条干净、可控的文本，再进行向量化与检索。


## 0) 导入依赖

确保已安装：`requests`, `numpy`, `faiss-cpu`, `sentence-transformers`, `jrag`。


In [1]:
import json
import time

import requests
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
import jrag

/home/miao/anaconda3/envs/tp-rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) 获取 JSON 数据（诺贝尔奖）

这里用公开 API 获取诺贝尔奖信息（`prizes` 列表）。


In [2]:
URL = "https://api.nobelprize.org/v1/prize.json"
headers = {"User-Agent": "Mozilla/5.0 (compatible; json-rag-notebook/1.0)"}

resp = requests.get(URL, timeout=30, headers=headers)
resp.raise_for_status()

data = resp.json()
assert isinstance(data, dict) and isinstance(data.get("prizes"), list)

prizes_lst = data["prizes"]

print("Successfully fetched and parsed JSON data.")
print(f"Number of events found: {len(prizes_lst)}")
print("\nFirst 3 records:\n", json.dumps(prizes_lst[:3], indent=2, ensure_ascii=False))

Successfully fetched and parsed JSON data.
Number of events found: 682

First 3 records:
 [
  {
    "year": "2025",
    "category": "chemistry",
    "laureates": [
      {
        "id": "1053",
        "firstname": "Susumu",
        "surname": "Kitagawa",
        "motivation": "\"for the development of metal–organic frameworks\"",
        "share": "3"
      },
      {
        "id": "1054",
        "firstname": "Richard",
        "surname": "Robson",
        "motivation": "\"for the development of metal–organic frameworks\"",
        "share": "3"
      },
      {
        "id": "1055",
        "firstname": "Omar M.",
        "surname": "Yaghi",
        "motivation": "\"for the development of metal–organic frameworks\"",
        "share": "3"
      }
    ]
  },
  {
    "year": "2025",
    "category": "economics",
    "overallMotivation": "\"for having explained innovation-driven economic growth\"",
    "laureates": [
      {
        "id": "1058",
        "firstname": "Joel",
        "surna

## 2) 预处理：把获奖者姓名合并成 `full_name`

同一份数据里，获奖者名字可能分散在 `firstname` 和 `surname` 两个字段。为了更好检索，这里合并出 `full_name`。


In [3]:
for prize_info in prizes_lst:
    laureates_list = prize_info.get("laureates", [])

    for laureate in laureates_list:
        if "firstname" in laureate and "surname" in laureate:
            first_name = laureate["firstname"]
            last_name = laureate["surname"]
            laureate["full_name"] = f"{first_name} {last_name}"
        elif "surname" not in laureate and "firstname" in laureate and len(laureate["firstname"].split(" ")) > 1:
            laureate["full_name"] = laureate["firstname"]

## 3) 用 `jrag` 把单条 JSON 记录转成“可 embed 的文本”

我们先用一条记录演示：

- `jrag_config`：key 是标签，value 是 `jsonpath-ng` 表达式，用来定位字段
- `jrag.to_text(record, config)`：抽取字段后拼成一条文本（这条文本将用于 embedding）


In [4]:
first_record = prizes_lst[0]

jrag_config = {
    "Year": "$.year",
    "Category": "$.category",
    "Laureats": "$.laureates[*].full_name",
    # 这里只取第一个 laureate 的 motivation（通常相同）
    "Motivation": "$.laureates[0].motivation",
}

example_text = jrag.to_text(first_record, jrag_config)
print(f"Text to embed:\n\"{example_text}\"")
print("\nBuilt from:")
print(json.dumps(first_record, indent=2))

Text to embed:
"Year: 2025 | Category: chemistry | Laureats: [Susumu Kitagawa, Richard Robson, Omar M. Yaghi] | Motivation: "for the development of metal–organic frameworks""

Built from:
{
  "year": "2025",
  "category": "chemistry",
  "laureates": [
    {
      "id": "1053",
      "firstname": "Susumu",
      "surname": "Kitagawa",
      "motivation": "\"for the development of metal\u2013organic frameworks\"",
      "share": "3",
      "full_name": "Susumu Kitagawa"
    },
    {
      "id": "1054",
      "firstname": "Richard",
      "surname": "Robson",
      "motivation": "\"for the development of metal\u2013organic frameworks\"",
      "share": "3",
      "full_name": "Richard Robson"
    },
    {
      "id": "1055",
      "firstname": "Omar M.",
      "surname": "Yaghi",
      "motivation": "\"for the development of metal\u2013organic frameworks\"",
      "share": "3",
      "full_name": "Omar M. Yaghi"
    }
  ]
}


## 4) 批量处理：给所有记录打上 `jrag_text`

`jrag.tag_list` 会对列表里每条 JSON 按配置抽取字段，并把拼接好的文本写入 `jrag_text` 字段。


In [5]:
jrag_config = {
    "Year": "$.year",
    "Category": "$.category",
    "Laureats": "$.laureates[*].full_name",
    "Motivation": "$.laureates[0].motivation",
}

prizes_tagged_lst = jrag.tag_list(prizes_lst, jrag_config)

# Inspect first example
print(prizes_tagged_lst[0]["jrag_text"])

Year: 2025 | Category: chemistry | Laureats: [Susumu Kitagawa, Richard Robson, Omar M. Yaghi] | Motivation: "for the development of metal–organic frameworks"


## 5) 构建语料（文本 + 元数据）

我们需要两份并行列表：

- `corpus_texts`：每条记录的 `jrag_text`（将被向量化）
- `corpus_metadata`：与文本一一对应的原始 JSON（方便检索后回溯原数据）


In [6]:
corpus_texts: list[str] = []
corpus_metadata: list[dict] = []

for i, item in enumerate(prizes_tagged_lst):
    jrag_text = item.get("jrag_text")
    if jrag_text and isinstance(jrag_text, str):
        corpus_texts.append(jrag_text)
        corpus_metadata.append({"original_index": i, "data": item})

assert corpus_texts, "No valid text content found in the JSON data."
print(f"Loaded {len(corpus_texts)} items with text content.")

Loaded 682 items with text content.


## 6) 生成 embeddings（Sentence-Transformers）

这里使用 `sentence-transformers` 的通用模型 `all-MiniLM-L6-v2`：速度快、效果稳定。

注意：语料 embedding 与 query embedding **必须使用同一个模型**，否则距离不可比。


In [7]:
import os
os.environ["HTTP_PROXY"] = "http://127.0.0.1:7890"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:7890"
os.environ["ALL_PROXY"] = "http://127.0.0.1:7890"

In [8]:
MODEL_NAME = "all-MiniLM-L6-v2"

print(f"Loading Sentence Transformer model '{MODEL_NAME}'...")
start_time = time.time()
model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded in {time.time() - start_time:.2f} seconds.")

print("Generating embeddings for the corpus...")
start_time = time.time()
corpus_embeddings = model.encode(corpus_texts, convert_to_numpy=True, show_progress_bar=True)
print(f"Embeddings generated in {time.time() - start_time:.2f} seconds.")

corpus_embeddings = corpus_embeddings.astype("float32")
embedding_dim = corpus_embeddings.shape[1]
print(f"Embedding dimension: {embedding_dim}")

Loading Sentence Transformer model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1615.22it/s]


Model loaded in 145.65 seconds.
Generating embeddings for the corpus...


Batches: 100%|██████████| 22/22 [00:00<00:00, 22.37it/s]

Embeddings generated in 1.00 seconds.
Embedding dimension: 384


## 7) 构建 FAISS 索引

我们用最基础的 `IndexFlatL2`（欧式距离，穷举搜索）。对小数据集非常直观：把所有向量放进索引，查询时找最近的 Top-K。


In [9]:
print("Building FAISS index (IndexFlatL2)...")
index = faiss.IndexFlatL2(embedding_dim)

print(f"Adding {len(corpus_embeddings)} embeddings to the index...")
index.add(corpus_embeddings)
print(f"Index contains {index.ntotal} vectors.")

Building FAISS index (IndexFlatL2)...
Adding 682 embeddings to the index...
Index contains 682 vectors.


## 8) 查询：把问题向量化，然后在 FAISS 里做 Top-K

流程：

- 把 `query_text` 编码为 query embedding
- `index.search(query_embedding, k)` 得到 `distances` 和 `indices`
- 用 `indices` 回到 `corpus_metadata` 找到对应的原始 JSON


In [10]:
query_text = "Breakthrough in medicine or physiology"
NUM_NEIGHBORS = 5

print(f"\nPerforming search for query: '{query_text}'")
print(f"Finding top {NUM_NEIGHBORS} similar items...")

start_time = time.time()
query_embedding = model.encode([query_text], convert_to_numpy=True).astype("float32")
print(f"Query embedding generated in {time.time() - start_time:.2f} seconds.")

start_time = time.time()
distances, indices = index.search(query_embedding, NUM_NEIGHBORS)
print(f"Search completed in {time.time() - start_time:.4f} seconds.")

print("\nSearch Results:")
print("--------------")

if not indices[0].size:
    print("No results found.")
else:
    for i, idx in enumerate(indices[0]):
        original_item_info = corpus_metadata[int(idx)]
        original_item = original_item_info["data"]
        distance = float(distances[0][i])

        print(f"Rank {i+1}:")
        print(f"  Distance: {distance:.4f}")
        print(f"  ID: {original_item.get('id', 'N/A')}")
        print(f"  Category: {original_item.get('category', 'N/A')}")
        print(f"  Content: {original_item}")
        print("-" * 10)


Performing search for query: 'Breakthrough in medicine or physiology'
Finding top 5 similar items...
Query embedding generated in 0.03 seconds.
Search completed in 0.0004 seconds.

Search Results:
--------------
Rank 1:
  Distance: 1.0166
  ID: N/A
  Category: medicine
  Content: {'year': '1999', 'category': 'medicine', 'laureates': [{'id': '461', 'firstname': 'Günter', 'surname': 'Blobel', 'motivation': '"for the discovery that proteins have intrinsic signals that govern their transport and localization in the cell"', 'share': '1', 'full_name': 'Günter Blobel'}], 'jrag_text': 'Year: 1999 | Category: medicine | Laureats: Günter Blobel | Motivation: "for the discovery that proteins have intrinsic signals that govern their transport and localization in the cell"'}
----------
Rank 2:
  Distance: 1.0256
  ID: N/A
  Category: medicine
  Content: {'year': '2000', 'category': 'medicine', 'laureates': [{'id': '722', 'firstname': 'Arvid', 'surname': 'Carlsson', 'motivation': '"for their discov